In [142]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

In [143]:

# 1. Load Ratings (UserID::MovieID::Rating::Timestamp)

rating_cols = ['user_id', 'item_id', 'rating', 'timestamp']
ratings = pd.read_csv('ml-1m/ratings.dat', sep='::', names=rating_cols, 
                      engine='python', encoding='latin-1')



In [144]:
ratings['rating'].value_counts()

rating
4    348971
3    261197
5    226310
2    107557
1     56174
Name: count, dtype: int64

In [145]:
# 2. Load Users (UserID::Gender::Age::Occupation::Zip-code)

user_cols = ['user_id', 'gender', 'age', 'occupation', 'zip']
users = pd.read_csv('ml-1m/users.dat', sep='::', names=user_cols, 
                    engine='python', encoding='latin-1')



In [146]:
users

,user_id,gender,age,occupation,zip
0,1,F,1,10,48067
1,2,M,56,16,70072
2,3,M,25,15,55117
3,4,M,45,7,02460
4,5,M,25,20,55455
...,...,...,...,...,...
6035,6036,F,25,15,32603
6036,6037,F,45,1,76006
6037,6038,F,56,1,14706
6038,6039,F,45,0,01060


In [147]:
users[users['age'] == 1] ### These are people below the age of 18, for more info about dataset -> readme

,user_id,gender,age,occupation,zip
0,1,F,1,10,48067
18,19,M,1,10,48073
50,51,F,1,10,10562
74,75,F,1,10,01748
85,86,F,1,10,54467
...,...,...,...,...,...
5843,5844,F,1,10,02131
5952,5953,M,1,10,21030
5972,5973,M,1,10,54701
5988,5989,F,1,10,74114


In [148]:
# 3. Load Movies (MovieID::Title::Genres)

movie_cols = ['item_id', 'title', 'genres']
movies = pd.read_csv('ml-1m/movies.dat', sep='::', names=movie_cols, 
                     engine='python', encoding='latin-1')

In [149]:
movies

,item_id,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy
...,...,...,...
3878,3948,Meet the Parents (2000),Comedy
3879,3949,Requiem for a Dream (2000),Drama
3880,3950,Tigerland (2000),Drama
3881,3951,Two Family House (2000),Drama


In [150]:
print(ratings.isnull().any())
print(movies.isnull().any())
print(users.isnull().any())

user_id      False
item_id      False
rating       False
timestamp    False
dtype: bool
item_id    False
title      False
genres     False
dtype: bool
user_id       False
gender        False
age           False
occupation    False
zip           False
dtype: bool


In [151]:
ratings['user_id']

0             1
1             1
2             1
3             1
4             1
           ... 
1000204    6040
1000205    6040
1000206    6040
1000207    6040
1000208    6040
Name: user_id, Length: 1000209, dtype: int64

###  Drop the original 'rating' to avoid accidental star-prediction logic, we need to only consider implicit interactions 

In [152]:
ratings['target'] = 1 

ratings = ratings.drop(columns=['rating'])

In [153]:
ratings['target']

0          1
1          1
2          1
3          1
4          1
          ..
1000204    1
1000205    1
1000206    1
1000207    1
1000208    1
Name: target, Length: 1000209, dtype: int64

In [154]:
users

,user_id,gender,age,occupation,zip
0,1,F,1,10,48067
1,2,M,56,16,70072
2,3,M,25,15,55117
3,4,M,45,7,02460
4,5,M,25,20,55455
...,...,...,...,...,...
6035,6036,F,25,15,32603
6036,6037,F,45,1,76006
6037,6038,F,56,1,14706
6038,6039,F,45,0,01060


In [155]:
print((movies['item_id'].unique()))
print(ratings['user_id'].unique())

[   1    2    3 ... 3950 3951 3952]
[   1    2    3 ... 6038 6039 6040]


### Mapping of user_to_index and item_to_index so that model won't select invalid(unmapped) indices i.e empty because these are identifiers just random numbers that's why adjusting them according to index

In [156]:
# Create mapping dictionaries
user_to_idx = {user_id: i for i, user_id in enumerate(ratings['user_id'].unique())}
item_to_idx = {item_id: i for i, item_id in enumerate(movies['item_id'].unique())}


# Apply to the ratings table
ratings['user_id'] = ratings['user_id'].map(user_to_idx)
ratings['item_id'] = ratings['item_id'].map(item_to_idx)

# # Save the counts for your model initialization
num_users = len(user_to_idx)
num_items = len(item_to_idx)

In [157]:
ratings

,user_id,item_id,timestamp,target
0,0,1176,978300760,1
1,0,655,978302109,1
2,0,902,978301968,1
3,0,3339,978300275,1
4,0,2286,978824291,1
...,...,...,...,...
1000204,6039,1075,956716541,1
1000205,6039,1078,956704887,1
1000206,6039,558,956704746,1
1000207,6039,1080,956715648,1


In [158]:
num_items,num_users

(3883, 6040)

In [159]:
ratings['item_id']

0          1176
1           655
2           902
3          3339
4          2286
           ... 
1000204    1075
1000205    1078
1000206     558
1000207    1080
1000208    1081
Name: item_id, Length: 1000209, dtype: int64

In [160]:
ratings

,user_id,item_id,timestamp,target
0,0,1176,978300760,1
1,0,655,978302109,1
2,0,902,978301968,1
3,0,3339,978300275,1
4,0,2286,978824291,1
...,...,...,...,...
1000204,6039,1075,956716541,1
1000205,6039,1078,956704887,1
1000206,6039,558,956704746,1
1000207,6039,1080,956715648,1


#### Leave-One-Out Temporal Split. A recommendation model can't look into the future. It has to predict what you will watch next based on what you had watched. Splitting by time prevents data leakage

In [161]:
# Sort by timestamp to find the latest interaction and insert it into test set 
ratings = ratings.sort_values(by=['user_id', 'timestamp'])

# Shift the last interaction for each user to the test set
test_ratings = ratings.groupby('user_id').tail(1)
train_ratings = ratings.drop(test_ratings.index)

print(f"Split complete. Training size: {len(train_ratings)}, Test size: {len(test_ratings)}")

Split complete. Training size: 994169, Test size: 6040


In [162]:
train_ratings

,user_id,item_id,timestamp,target
31,0,3117,978300019,1
22,0,1250,978300055,1
27,0,1672,978300055,1
37,0,1009,978300055,1
24,0,2271,978300103,1
...,...,...,...,...
999923,6039,229,997454398,1
1000019,6039,2848,997454429,1
999988,6039,1852,997454464,1
1000172,6039,1726,997454464,1


In [163]:
ratings

,user_id,item_id,timestamp,target
31,0,3117,978300019,1
22,0,1250,978300055,1
27,0,1672,978300055,1
37,0,1009,978300055,1
24,0,2271,978300103,1
...,...,...,...,...
1000019,6039,2848,997454429,1
999988,6039,1852,997454464,1
1000172,6039,1726,997454464,1
1000167,6039,159,997454486,1


In [164]:
train_ratings

,user_id,item_id,timestamp,target
31,0,3117,978300019,1
22,0,1250,978300055,1
27,0,1672,978300055,1
37,0,1009,978300055,1
24,0,2271,978300103,1
...,...,...,...,...
999923,6039,229,997454398,1
1000019,6039,2848,997454429,1
999988,6039,1852,997454464,1
1000172,6039,1726,997454464,1


In [165]:
num_items # There are 6040 unique users and 3706 movies

3883

In [166]:
movies

,item_id,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy
...,...,...,...
3878,3948,Meet the Parents (2000),Comedy
3879,3949,Requiem for a Dream (2000),Drama
3880,3950,Tigerland (2000),Drama
3881,3951,Two Family House (2000),Drama


In [167]:
len(movies)

3883

#### Creating a lookup for positive and negative instances 0's and 1's. here we use negative sampling strategy of 4 : 1. So for every user we map it with for 4 random movies that they never interacted with. It's an unobserved entity from the user perspective

In [168]:
### negative sampling function is transferred to train.py 


In [169]:
train_ratings[['item_id','user_id']]

,item_id,user_id
31,3117,0
22,1250,0
27,1672,0
37,1009,0
24,2271,0
...,...,...
999923,229,6039
1000019,2848,6039
999988,1852,6039
1000172,1726,6039


In [170]:
train_ratings

,user_id,item_id,timestamp,target
31,0,3117,978300019,1
22,0,1250,978300055,1
27,0,1672,978300055,1
37,0,1009,978300055,1
24,0,2271,978300103,1
...,...,...,...,...
999923,6039,229,997454398,1
1000019,6039,2848,997454429,1
999988,6039,1852,997454464,1
1000172,6039,1726,997454464,1


In [171]:

# Generate the data for the training loop
# train_u, train_i, train_labels = get_train_instances(train_ratings,num_items)

In [172]:
# train_u,train_i,train_labels

In [173]:
users

,user_id,gender,age,occupation,zip
0,1,F,1,10,48067
1,2,M,56,16,70072
2,3,M,25,15,55117
3,4,M,45,7,02460
4,5,M,25,20,55455
...,...,...,...,...,...
6035,6036,F,25,15,32603
6036,6037,F,45,1,76006
6037,6038,F,56,1,14706
6038,6039,F,45,0,01060


In [174]:
users['user_id']
users['user_id'] = users['user_id'].map(user_to_idx) # mapping in user dataframe

In [175]:
users['gender'] = users['gender'].map({'M': 0, 'F': 1})
# users['gender'].value_counts()
 
# users['gender'] = users['gender'].str.strip()

In [176]:
users

,user_id,gender,age,occupation,zip
0,0,1,1,10,48067
1,1,0,56,16,70072
2,2,0,25,15,55117
3,3,0,45,7,02460
4,4,0,25,20,55455
...,...,...,...,...,...
6035,6035,1,25,15,32603
6036,6036,1,45,1,76006
6037,6037,1,56,1,14706
6038,6038,1,45,0,01060


In [177]:
users['gender'].value_counts()

gender
0    4331
1    1709
Name: count, dtype: int64

In [178]:
users['occupation'].value_counts()

occupation
4     759
0     711
7     679
1     528
17    502
12    388
14    302
20    281
2     267
16    241
6     236
10    195
3     173
15    144
13    142
11    129
5     112
9      92
19     72
18     70
8      17
Name: count, dtype: int64

In [179]:
users.columns

Index(['user_id', 'gender', 'age', 'occupation', 'zip'], dtype='object')

In [180]:
users['zip'].value_counts()

zip
48104    19
22903    18
94110    17
55104    17
55105    16
         ..
91330     1
58102     1
91306     1
61265     1
30066     1
Name: count, Length: 3439, dtype: int64

In [181]:
item_to_idx

{np.int64(1): 0,
 np.int64(2): 1,
 np.int64(3): 2,
 np.int64(4): 3,
 np.int64(5): 4,
 np.int64(6): 5,
 np.int64(7): 6,
 np.int64(8): 7,
 np.int64(9): 8,
 np.int64(10): 9,
 np.int64(11): 10,
 np.int64(12): 11,
 np.int64(13): 12,
 np.int64(14): 13,
 np.int64(15): 14,
 np.int64(16): 15,
 np.int64(17): 16,
 np.int64(18): 17,
 np.int64(19): 18,
 np.int64(20): 19,
 np.int64(21): 20,
 np.int64(22): 21,
 np.int64(23): 22,
 np.int64(24): 23,
 np.int64(25): 24,
 np.int64(26): 25,
 np.int64(27): 26,
 np.int64(28): 27,
 np.int64(29): 28,
 np.int64(30): 29,
 np.int64(31): 30,
 np.int64(32): 31,
 np.int64(33): 32,
 np.int64(34): 33,
 np.int64(35): 34,
 np.int64(36): 35,
 np.int64(37): 36,
 np.int64(38): 37,
 np.int64(39): 38,
 np.int64(40): 39,
 np.int64(41): 40,
 np.int64(42): 41,
 np.int64(43): 42,
 np.int64(44): 43,
 np.int64(45): 44,
 np.int64(46): 45,
 np.int64(47): 46,
 np.int64(48): 47,
 np.int64(49): 48,
 np.int64(50): 49,
 np.int64(51): 50,
 np.int64(52): 51,
 np.int64(53): 52,
 np.int64(54

In [182]:
movies['item_id']

0          1
1          2
2          3
3          4
4          5
        ... 
3878    3948
3879    3949
3880    3950
3881    3951
3882    3952
Name: item_id, Length: 3883, dtype: int64

In [183]:
movies['item_id'] = movies['item_id'].map(item_to_idx)

In [184]:
movies['item_id']

0          0
1          1
2          2
3          3
4          4
        ... 
3878    3878
3879    3879
3880    3880
3881    3881
3882    3882
Name: item_id, Length: 3883, dtype: int64

In [185]:
user_to_idx

{np.int64(1): 0,
 np.int64(2): 1,
 np.int64(3): 2,
 np.int64(4): 3,
 np.int64(5): 4,
 np.int64(6): 5,
 np.int64(7): 6,
 np.int64(8): 7,
 np.int64(9): 8,
 np.int64(10): 9,
 np.int64(11): 10,
 np.int64(12): 11,
 np.int64(13): 12,
 np.int64(14): 13,
 np.int64(15): 14,
 np.int64(16): 15,
 np.int64(17): 16,
 np.int64(18): 17,
 np.int64(19): 18,
 np.int64(20): 19,
 np.int64(21): 20,
 np.int64(22): 21,
 np.int64(23): 22,
 np.int64(24): 23,
 np.int64(25): 24,
 np.int64(26): 25,
 np.int64(27): 26,
 np.int64(28): 27,
 np.int64(29): 28,
 np.int64(30): 29,
 np.int64(31): 30,
 np.int64(32): 31,
 np.int64(33): 32,
 np.int64(34): 33,
 np.int64(35): 34,
 np.int64(36): 35,
 np.int64(37): 36,
 np.int64(38): 37,
 np.int64(39): 38,
 np.int64(40): 39,
 np.int64(41): 40,
 np.int64(42): 41,
 np.int64(43): 42,
 np.int64(44): 43,
 np.int64(45): 44,
 np.int64(46): 45,
 np.int64(47): 46,
 np.int64(48): 47,
 np.int64(49): 48,
 np.int64(50): 49,
 np.int64(51): 50,
 np.int64(52): 51,
 np.int64(53): 52,
 np.int64(54

In [186]:
users['user_id']

0          0
1          1
2          2
3          3
4          4
        ... 
6035    6035
6036    6036
6037    6037
6038    6038
6039    6039
Name: user_id, Length: 6040, dtype: int64

In [187]:
movies['genres'].value_counts()

genres
Drama                              843
Comedy                             521
Horror                             178
Comedy|Drama                       162
Comedy|Romance                     142
                                  ... 
Drama|Film-Noir                      1
Comedy|Horror|Sci-Fi                 1
Adventure|Drama|Romance|Sci-Fi       1
Adventure|Animation|Sci-Fi           1
Adventure|Crime|Sci-Fi|Thriller      1
Name: count, Length: 301, dtype: int64

In [188]:
movies['genre_list'] = movies['genres'].apply(lambda x: x.split('|'))

In [189]:
movies['genre_list']

0        [Animation, Children's, Comedy]
1       [Adventure, Children's, Fantasy]
2                      [Comedy, Romance]
3                        [Comedy, Drama]
4                               [Comedy]
                      ...               
3878                            [Comedy]
3879                             [Drama]
3880                             [Drama]
3881                             [Drama]
3882                   [Drama, Thriller]
Name: genre_list, Length: 3883, dtype: object

In [190]:
from sklearn.preprocessing import MultiLabelBinarizer
mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(movies['genre_list'])
num_genres = len(mlb.classes_)

In [191]:
num_items,num_genres,num_users

(3883, 18, 6040)

In [192]:
mlb.classes_


array(['Action', 'Adventure', 'Animation', "Children's", 'Comedy',
       'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror',
       'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War',
       'Western'], dtype=object)

In [193]:
users

,user_id,gender,age,occupation,zip
0,0,1,1,10,48067
1,1,0,56,16,70072
2,2,0,25,15,55117
3,3,0,45,7,02460
4,4,0,25,20,55455
...,...,...,...,...,...
6035,6035,1,25,15,32603
6036,6036,1,45,1,76006
6037,6037,1,56,1,14706
6038,6038,1,45,0,01060


In [194]:
user_meta_df = users.sort_values('user_id')[['gender', 'age', 'occupation']]

In [195]:
genre_df = pd.DataFrame(
    genre_matrix,
    columns=mlb.classes_,   # e.g. ['Action', 'Comedy', 'Drama', ...]
    index=movies.index
)

In [196]:
genre_df

,Action,Adventure,Animation,Children's,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0
2,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0
3,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3878,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
3879,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
3880,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
3881,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0


In [197]:
movies = pd.concat([movies, genre_df], axis=1)

In [198]:
movies

,item_id,title,genres,genre_list,Action,Adventure,Animation,Children's,Comedy,Crime,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,0,Toy Story (1995),Animation|Children's|Comedy,"[Animation, Children's, Comedy]",0,0,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
1,1,Jumanji (1995),Adventure|Children's|Fantasy,"[Adventure, Children's, Fantasy]",0,1,0,1,0,0,...,1,0,0,0,0,0,0,0,0,0
2,2,Grumpier Old Men (1995),Comedy|Romance,"[Comedy, Romance]",0,0,0,0,1,0,...,0,0,0,0,0,1,0,0,0,0
3,3,Waiting to Exhale (1995),Comedy|Drama,"[Comedy, Drama]",0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
4,4,Father of the Bride Part II (1995),Comedy,[Comedy],0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3878,3878,Meet the Parents (2000),Comedy,[Comedy],0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
3879,3879,Requiem for a Dream (2000),Drama,[Drama],0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3880,3880,Tigerland (2000),Drama,[Drama],0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3881,3881,Two Family House (2000),Drama,[Drama],0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [199]:
genre_matrix

array([[0, 0, 1, ..., 0, 0, 0],
       [0, 1, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 1, 0, 0]], shape=(3883, 18))

In [200]:
train_ratings

,user_id,item_id,timestamp,target
31,0,3117,978300019,1
22,0,1250,978300055,1
27,0,1672,978300055,1
37,0,1009,978300055,1
24,0,2271,978300103,1
...,...,...,...,...
999923,6039,229,997454398,1
1000019,6039,2848,997454429,1
999988,6039,1852,997454464,1
1000172,6039,1726,997454464,1


In [201]:
idx_to_user = {idx: raw_id for raw_id, idx in user_to_idx.items()}
idx_to_item = {idx: raw_id for raw_id, idx in item_to_idx.items()}

In [202]:
movies

,item_id,title,genres,genre_list,Action,Adventure,Animation,Children's,Comedy,Crime,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,0,Toy Story (1995),Animation|Children's|Comedy,"[Animation, Children's, Comedy]",0,0,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
1,1,Jumanji (1995),Adventure|Children's|Fantasy,"[Adventure, Children's, Fantasy]",0,1,0,1,0,0,...,1,0,0,0,0,0,0,0,0,0
2,2,Grumpier Old Men (1995),Comedy|Romance,"[Comedy, Romance]",0,0,0,0,1,0,...,0,0,0,0,0,1,0,0,0,0
3,3,Waiting to Exhale (1995),Comedy|Drama,"[Comedy, Drama]",0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
4,4,Father of the Bride Part II (1995),Comedy,[Comedy],0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3878,3878,Meet the Parents (2000),Comedy,[Comedy],0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
3879,3879,Requiem for a Dream (2000),Drama,[Drama],0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3880,3880,Tigerland (2000),Drama,[Drama],0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3881,3881,Two Family House (2000),Drama,[Drama],0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [203]:
# Assuming your preprocessed movies DataFrame has 'item_id' (0 to N-1) and 'title'
idx_to_title = dict(zip(movies['item_id'], movies['title']))

In [204]:
idx_to_title

{0: 'Toy Story (1995)',
 1: 'Jumanji (1995)',
 2: 'Grumpier Old Men (1995)',
 3: 'Waiting to Exhale (1995)',
 4: 'Father of the Bride Part II (1995)',
 5: 'Heat (1995)',
 6: 'Sabrina (1995)',
 7: 'Tom and Huck (1995)',
 8: 'Sudden Death (1995)',
 9: 'GoldenEye (1995)',
 10: 'American President, The (1995)',
 11: 'Dracula: Dead and Loving It (1995)',
 12: 'Balto (1995)',
 13: 'Nixon (1995)',
 14: 'Cutthroat Island (1995)',
 15: 'Casino (1995)',
 16: 'Sense and Sensibility (1995)',
 17: 'Four Rooms (1995)',
 18: 'Ace Ventura: When Nature Calls (1995)',
 19: 'Money Train (1995)',
 20: 'Get Shorty (1995)',
 21: 'Copycat (1995)',
 22: 'Assassins (1995)',
 23: 'Powder (1995)',
 24: 'Leaving Las Vegas (1995)',
 25: 'Othello (1995)',
 26: 'Now and Then (1995)',
 27: 'Persuasion (1995)',
 28: 'City of Lost Children, The (1995)',
 29: 'Shanghai Triad (Yao a yao yao dao waipo qiao) (1995)',
 30: 'Dangerous Minds (1995)',
 31: 'Twelve Monkeys (1995)',
 32: 'Wings of Courage (1995)',
 33: 'Babe (19

In [205]:
import pickle
import os

# Create data output directory
os.makedirs("data", exist_ok=True)

# Align User Metadata DataFrame to mapped user_id indices (0..6039)
user_meta_df = users.sort_values('user_id')[['gender', 'age', 'occupation']].reset_index(drop=True)

# 1. Package all essentials into a dictionary
preprocessed_artifacts = {
    # Data DataFrames / Arrays
    'train_ratings': train_ratings[['user_id', 'item_id', 'target']], # Training split DataFrame
    'test_ratings': test_ratings[['user_id', 'item_id', 'target']],   # Test split DataFrame
    'user_meta_df': user_meta_df,                                     # Demographic feature table
    'item_genre_matrix': genre_matrix,                           # Multi-hot genre array (3706, 18)
    
    # Metadata Mappings & Counts
    'idx_to_user' : idx_to_user,
    'idx_to_item' : idx_to_item,
    'idx_to_title': idx_to_title,
    'user_to_idx': user_to_idx,
    'item_to_idx': item_to_idx,
    'num_users': num_users,                                           # 6040
    'num_items': num_items,                                           # 3706
    'num_genres': num_genres                                           # 18
}

# 2. Save artifact file to disk
with open("data/movielens_preprocessed.pkl", "wb") as f:
    pickle.dump(preprocessed_artifacts, f)

print("Preprocessed essentials successfully saved to data/movielens_preprocessed.pkl!")

Preprocessed essentials successfully saved to data/movielens_preprocessed.pkl!
